# 04 — Understandability (ISO/IEC 25010)

**Auteur :** Benoît Moraillon — **Licence :** AGPL-3.0

L'**understandability** est la métrique phare de KodoNeko. Elle synthétise dans un **niveau de risque** 0-100 la compréhensibilité du code, en combinant six dimensions complémentaires.

## Vocabulaire : risk, pas score

Dans KodoNeko, on distingue strictement deux notions :

| Notion | Orientation | Où la trouver |
|---|---|---|
| **risk** | *Pire si plus haut* | `UnderstandabilityMetrics.risk` (cette métrique) |
| **score** | *Mieux si plus haut* | `PrimaryQualityScore.score` (notebook 05) |

Cette asymétrie est volontaire : un *risque* élevé est mauvais ; un *score* élevé est bon. Ne pas confondre.

## Référence normative

**ISO/IEC 25010:2011** définit *Understandability* comme une sous-caractéristique d'*Usability* : « degree to which a product or system has attributes that enable users to easily understand whether it is appropriate ».

C'est exactement ce qu'on veut mesurer : un humain qui découvre ce code, à quelle vitesse va-t-il le comprendre ?

## Pourquoi un risque composite

Aucune métrique simple ne capture seule la compréhensibilité :

- Une fonction de 200 lignes peut être lisible si elle est plate (mais elle est longue).
- Une fonction de 10 lignes peut être horrible si elle est dense, mal nommée, et imbrique 5 conditions.
- Un nom de variable `x` au lieu de `customer_age` change drastiquement la difficulté.

L'understandability **combine** six dimensions en un risque unique.

## Les six sous-métriques

| Sous-métrique | Poids | Mesure |
|---|---|---|
| `cognitive` | 30 % | Complexité cognitive de Campbell (2017) — pénalise l'imbrication |
| `function_length` | 20 % | Longueur max des fonctions du fichier |
| `nesting_depth` | 15 % | Profondeur d'imbrication maximale |
| `parameter_count` | 15 % | Nombre max de paramètres trouvé |
| `line_density` | 10 % | Caractères non-blancs / ligne (densité visuelle) |
| `identifier_length` | 10 % | Longueur moyenne des identifiants (**inversé** : plus court = pire) |

Chaque sous-métrique est normalisée 0-100 selon trois seuils (`low`, `high`, `very_high`), puis combinée linéairement.

## Niveaux qualitatifs

| Risk | Niveau | Interprétation |
|---|---|---|
| 0-10 | `low` | Code clair |
| 11-27 | `moderate` | Lecture attentive nécessaire |
| 28-50 | `high` | Difficile — refactoring conseillé |
| > 50 | `very_high` | Très difficile — refactoring fortement conseillé |

## Cognitive complexity (Campbell) intégrée

La cognitive complexity (G. Ann Campbell, 2017) est **conservée comme sous-attribut** (`understandability.cognitive`) pour préserver la comparabilité avec radon et SonarSource — outils standards de l'écosystème qualité logicielle.

In [ ]:
# Bootstrap : permet l'import des modules sans installation pip
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import setup_paths
setup_paths.setup()

## Progression : trois exemples, du sain au critique

On va analyser trois fonctions qui forment une progression nette du risque. À chaque étape, on liste les dimensions qui poussent le risk vers le haut.

### Exemple 1 — Sain (`low`)

Courte, plate, nommée clairement.

In [ ]:
from kodoneko_metrics import count_understandability

sain = """
def somme_positive(values):
    \"\"\"Retourne la somme des valeurs positives.\"\"\"
    return sum(v for v in values if v > 0)
"""

u = count_understandability(sain, language="python")
print(f"Risk : {u.risk:.1f}/100  ({u.level})")
print(f"  cognitive          : {u.cognitive}")
print(f"  function_length_max: {u.function_length_max}")
print(f"  nesting_depth_max  : {u.nesting_depth_max}")
print(f"  parameter_count_max: {u.parameter_count_max}")
print(f"  identifier_length  : {u.identifier_length_avg:.1f}")

**Lecture** : aucune dimension ne dépasse le palier `low`. Risque très faible, comme attendu.

### Exemple 2 — Le piège « cognitive faible ≠ code lisible »

C'est l'exemple emblématique qui justifie la fusion v2.0. Cette fonction n'a **aucune imbrication** — sa cognitive complexity (Campbell) est donc quasi nulle. Pourtant le code est pénible à lire : longueur excessive, identifiants 1 caractère, expressions denses.

Avant la v2.0 (cognitive seule), ce code aurait été qualifié de "lisible" alors qu'aucun humain ne le qualifierait ainsi. L'understandability le détecte parce qu'elle **agrège plusieurs dimensions**.

In [ ]:
piege_cognitive_faible = """
def p(a, b, c, d, e, f, g, h):
    x1 = (a + 1) * (b - 2) / (c + 3)
    x2 = (a + 2) * (b - 3) / (c + 4)
    x3 = (a + 3) * (b - 4) / (c + 5)
    x4 = (d + 1) * (e - 2) / (f + 3)
    x5 = (d + 2) * (e - 3) / (f + 4)
    x6 = (d + 3) * (e - 4) / (f + 5)
    y1 = (g + 1) * (h - 2) / (a + 3)
    y2 = (g + 2) * (h - 3) / (a + 4)
    y3 = (g + 3) * (h - 4) / (a + 5)
    y4 = (a + b) * (c + d) / (e + f)
    y5 = (a - b) * (c - d) / (e - f)
    y6 = (a * b) + (c * d) - (e * f)
    z1 = x1 + x2 + x3 + x4 + x5 + x6
    z2 = y1 + y2 + y3 + y4 + y5 + y6
    z3 = (z1 + z2) * (a + b + c + d + e + f + g + h)
    z4 = (z1 - z2) * (a - b + c - d + e - f + g - h)
    z5 = z3 * z4 / (z1 + z2 + 1)
    z6 = (z3 + z4) / (z5 + 1)
    z7 = z1 * z2 * z3 * z4 * z5 * z6
    z8 = (z7 + 1) / (z6 + 1) * (z5 + 1) / (z4 + 1)
    return z8 + z7 + z6 + z5 + z4 + z3 + z2 + z1
"""

u = count_understandability(piege_cognitive_faible, language="python")
print(f"Risk : {u.risk:.1f}/100  ({u.level})")
print()
print("Décomposition :")
print(f"  cognitive          : {u.cognitive} ← très faible (pas d'imbrication)")
print(f"  function_length_max: {u.function_length_max} lignes ← long")
print(f"  parameter_count_max: {u.parameter_count_max} ← beaucoup")
print(f"  line_density_avg   : {u.line_density_avg:.1f} ← dense")
print(f"  identifier_length  : {u.identifier_length_avg:.1f} ← très courts")

**Lecture cruciale** : la cognitive seule de Campbell reste basse parce qu'il n'y a aucune imbrication. Avant la fusion v2.0, ce fichier serait passé inaperçu côté cognitive. Mais l'understandability **agrège tout** :

- longueur très au-dessus de `function_length.high` (50 lignes)
- 8 paramètres (saturé sur `parameter_count`)
- identifiants 1 caractère (saturé sur `identifier_length`)
- densité visuelle élevée

Plusieurs composantes pénalisent simultanément → le risk monte en `moderate`.

C'est exactement la valeur de la fusion : ne plus laisser une seule dimension dicter le verdict.

### Exemple 3 — Critique (`high` ou `very_high`)

Ici toutes les dimensions sont dégradées **en même temps**, y compris l'imbrication profonde. Le risk atteint le palier critique.

In [ ]:
critique = """
def f(a, b, c, d, e, g, h):
    r = 0
    s = 0
    t = 0
    if a > 0:
        for i in range(b):
            if c[i]:
                for j in range(d):
                    if e[j]:
                        if g[i][j]:
                            r += h * (a + b) * (c[i] - d) / (e[j] + 1)
                        elif h[i][j] < 0:
                            s -= a * b + c[i] * d - e[j]
                        else:
                            t += (a + b + c[i] + d + e[j] + g[i][j])
                    elif d > 0:
                        if c[i] > e[0]:
                            r += a * d
                        else:
                            s += b * d
            elif b > c[0]:
                t += a + b + c[0]
        for k in range(b):
            if c[k] and e[k]:
                r += g[k][0] * h[k][0]
                s -= g[k][0] - h[k][0]
                t *= (g[k][0] + h[k][0]) / (a + 1)
    return r + s + t
"""

u = count_understandability(critique, language="python")
print(f"Risk : {u.risk:.1f}/100  ({u.level})")
print()
print("Décomposition :")
print(f"  cognitive          : {u.cognitive} ← très élevée (imbrication profonde)")
print(f"  function_length_max: {u.function_length_max}")
print(f"  nesting_depth_max  : {u.nesting_depth_max} ← imbrication profonde")
print(f"  parameter_count_max: {u.parameter_count_max}")
print(f"  identifier_length  : {u.identifier_length_avg:.1f}")

**Lecture** : toutes les dimensions saturent ensemble. C'est le profil typique d'une fonction à refactorer en priorité.

### Comparaison synthétique

In [ ]:
import textwrap

print(f"{'Exemple':<35} {'Risk':>6}  {'Level':<10}  {'Cognitive':<10}")
print("-" * 70)
for nom, src in [
    ("1. Sain", sain),
    ("2. Cognitive faible, code dense", piege_cognitive_faible),
    ("3. Critique (toutes dimensions)", critique),
]:
    u = count_understandability(src, language="python")
    print(f"{nom:<35} {u.risk:>6.1f}  {u.level:<10}  {u.cognitive}")

**Observation clé** : entre l'exemple 1 (cognitive=1) et l'exemple 2 (cognitive=0), la cognitive **baisse** alors que le risk d'understandability **monte**. C'est la preuve que les deux signaux ne disent pas la même chose, et qu'agréger les six dimensions évite de passer à côté du genre de code que l'exemple 2 représente.

## Détail par fonction

`count_understandability_by_function` retourne **uniquement la liste des fonctions**, triée par risque décroissant. Utile quand un fichier global est bon mais qu'une fonction problématique s'y cache.

In [ ]:
from kodoneko_metrics import count_understandability_by_function

fichier_mixte = """
def helper_propre():
    return 42

def autre_propre(x):
    return x * 2

def fonction_problematique(a, b, c, d, e):
    r = 0
    if a:
        for i in range(b):
            if c[i]:
                for j in range(d):
                    if e[j]:
                        r += 1
    return r

def encore_proprette():
    return None
"""

funcs = count_understandability_by_function(fichier_mixte, language="python")
print(f"{'Fonction':<30} {'Risk':>6}  {'Level':<10} {'Lignes':>7} {'Params':>7}")
print("-" * 70)
for f in funcs:
    print(f"{f.name:<30} {f.risk:>6.1f}  {f.level:<10} {f.length:>7} {f.parameter_count:>7}")

La liste est déjà triée : **la fonction au risque le plus élevé apparaît en premier**, indépendamment de la moyenne du fichier. Pratique pour cibler le refactoring.

## Récapitulatif

| Fonction | Retour |
|---|---|
| `count_understandability(source, language=...)` | `UnderstandabilityMetrics` (agrégat fichier + détail par fonction) |
| `count_understandability_by_function(source, ...)` | `list[FunctionUnderstandability]` triée par `.risk` décroissant |

Et dans le scanner (notebook 05), c'est cette métrique qui pèse **70 %** du PrimaryQualityScore.

---

# Mesurer sur un dépôt réel, et dans le temps

La compréhensibilité agrège plusieurs signaux (complexité, charge cognitive,
taille, imbrication) en un score composite inspiré d'ISO/IEC 25010. C'est la
métrique la plus proche du ressenti d'un développeur qui ouvre un fichier
inconnu. Mesurons-la sur un dépôt réel, puis suivons sa dérive — car un code
se dégrade rarement d'un coup, mais par petites érosions successives.

> ⚠️ Cette section clone un dépôt public (nécessite un accès réseau) et, pour
> les langages autres que Python, requiert `tree-sitter`. Sans ces prérequis,
> les cellules affichent un message explicatif et s'interrompent proprement,
> sans erreur.

## 1. Récupérer un dépôt public

Nous clonons un petit projet open-source réel. Le clone est *idempotent* : si le
dépôt est déjà présent localement, on le réutilise sans le retélécharger.

In [ ]:
import subprocess, tempfile
from pathlib import Path

REPO_URL = "https://github.com/gothinkster/django-realworld-example-app.git"
repo_dir = Path(tempfile.gettempdir()) / "kodoneko_demo_repo"

if repo_dir.exists():
    print("Dépôt déjà présent :", repo_dir)
else:
    print("Clonage en cours...")
    res = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print("Cloné :", repo_dir)
    else:
        print("Clone impossible (réseau indisponible ?) :")
        print("  ", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "erreur inconnue")
        repo_dir = None

## 2. Analyser le dépôt complet (instantané)

Premier réflexe : prendre une photographie de l'état actuel. On mesure la compréhensibilité (ISO/IEC 25010)
sur l'ensemble du dépôt, tel qu'il est à l'instant présent (le dernier commit).

In [ ]:
from kodoneko_scanner import scan_repo

if repo_dir:
    report = scan_repo(repo_dir)
    print(f"Fichiers analysés : {report.files_scanned}")
    print(f"  Compréhensibilité moyenne : {report.totals.understandability_avg:.1f}/100")
    print(f"  Pic de difficulté (max)   : {report.totals.understandability_max:.1f}")
    print(f"  Charge cognitive moyenne  : {report.totals.cognitive_avg:.1f}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 3. Analyser un commit précis

Le moteur temporel `kodoneko_temporal` sait reconstituer l'état du dépôt à
**n'importe quelle référence git** (un tag, une branche, un SHA, ou une
expression comme `HEAD~5`). Ici, on mesure compréhensibilité à ce point de l'histoire, cinq commits
en arrière — un véritable voyage dans le passé du code.

In [ ]:
from kodoneko_temporal import analyze_at_ref

def mesure(path):
    """Applique notre métrique à un état du dépôt."""
    report = scan_repo(path)
    return round(report.totals.understandability_avg, 1)

if repo_dir:
    try:
        point = analyze_at_ref(repo_dir, "HEAD~5", analyzer=mesure)
        print(f"À {point.label} (commit {point.sha[:8]}) : compréhensibilité = {point.result}")
    except Exception as e:
        print(f"Commit indisponible (historique trop court ?) : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 4. Mesurer un écart entre deux moments (*diff temporel*)

La question la plus parlante n'est pas « combien ? » mais « **combien de plus
qu'avant ?** ». En comparant la mesure à deux références, on quantifie ce qu'un
intervalle de développement a réellement produit comme compréhensibilité.

In [ ]:
if repo_dir:
    try:
        avant = analyze_at_ref(repo_dir, "HEAD~10", analyzer=mesure)
        apres = analyze_at_ref(repo_dir, "HEAD", analyzer=mesure)
        delta = apres.result - avant.result
        signe = "+" if delta >= 0 else ""
        # Variation relative en pourcentage (2 décimales), protégée contre la division par zéro
        if avant.result:
            pct = delta / avant.result * 100
            pct_txt = f"{signe}{pct:.2f} %"
        else:
            pct_txt = "n/a (valeur initiale nulle)"
        print(f"compréhensibilité il y a 10 commits : {avant.result}")
        print(f"compréhensibilité aujourd'hui       : {apres.result}")
        print(f"Variation absolue    : {signe}{delta}")
        print(f"Variation relative   : {pct_txt}")
    except Exception as e:
        print(f"Diff indisponible : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

### L'érosion lente de la lisibilité

Aucun commit ne rend un code illisible à lui seul. C'est l'accumulation qui
use. En traçant la compréhensibilité moyenne dans le temps, on rend visible
cette érosion lente, et on peut agir avant que la dette ne devienne ingérable.

In [ ]:
from kodoneko_temporal import analyze_over_windows

if repo_dir:
    serie = analyze_over_windows(repo_dir, analyzer=mesure, strategy="monthly")
    print(f"compréhensibilité à la fin de chaque mois ({len(serie)} fenêtres) :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    precedent = None
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        # Variation par rapport au mois précédent, en % (2 décimales)
        if precedent is None:
            var_txt = "     —  "
        elif precedent:
            pct = (point.result - precedent) / precedent * 100
            var_txt = f"{pct:+6.2f} %"
        else:
            var_txt = "    n/a "
        print(f"  {point.label}  {point.result:>8}  {var_txt}  {barre}")
        precedent = point.result
else:
    print("(dépôt indisponible — étape ignorée)")

## En résumé

Nous venons de parcourir les quatre façons d'interroger la compréhensibilité (ISO/IEC 25010) :

1. **L'instantané** — l'état présent du dépôt entier.
2. **Le commit précis** — la mesure à un point quelconque de l'histoire.
3. **Le diff temporel** — l'écart entre deux moments, qui isole ce qu'une
   période a produit.
4. **La série temporelle** — la trajectoire complète, qui révèle les tendances.

Le même analyseur (`mesure`) a servi pour les quatre : c'est toute la force du
moteur temporel, qui découple *ce qu'on mesure* de *quand on le mesure*.